In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
from scipy.stats import zscore
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from IPython.display import clear_output

## SysBio Blood Transcriptomics Sample Harmonization and PCA Plot Creation
### Harmonizing Blood Transcriptomics from AMP PD and AMP CMD

In [ ]:
# Paths to SysBio blood AMP PD and AMP CMD transcriptomics datasets
CMD_df = pd.read_csv("sysbio/Transcriptomics/Blood/CMD/GSE118165_raw_counts_GRCh38.p13_NCBI 2.tsv", sep = "\t", engine='c')
PD_df= pd.read_csv("sysbio/Transcriptomics/Blood/PD/releases_2023_v4release_1027_rnaseq-WB-RWTS-VHBS_subread_feature-counts_matrix.featureCounts.tsv", sep = "\t", engine='c')
cmd_samples = pd.read_csv("sysbio/Transcriptomics/Blood/CMD/Sample_Data.csv", engine='c')
cmd_gene_Ann = pd.read_csv("sysbio/Transcriptomics/Blood/CMD/Human.GRCh38.p13.annot 2.tsv", sep = "\t", engine='c')

In [ ]:
# View AMP PD dataset
PD_df

In [ ]:
# Rename samples
id_to_name = dict(zip(cmd_samples['Sample_ID'], cmd_samples['Sample_Name']))
CMD_df = CMD_df.rename(columns=id_to_name)
print(CMD_df.head())

      GeneID  1001-Bulk_B-U  1001-Bulk_B-S  1001-Mem_B-U  1001-Mem_B-S  \
0  100287102              5              1             3             1   
1     653635            433            167           307           210   
2  102466751              7              3             3             1   
3  107985730              0              0             2             0   
4  100302278              0              0             0             0   

   1001-Naive_B-U  1001-Naive_B-S  1001-Plasmablasts-U  1001-CD8pos_T-U  \
0               0               0                    0                1   
1             474             243                   47              362   
2               0               0                    0                2   
3               0               0                    0                0   
4               0               0                    0                0   

   1001-CD8pos_T-S  ...  GSM3319899  GSM3319900  GSM3319901  GSM3319902  \
0               24  ...      

In [ ]:
# Merge AMP CMD samples with gene annotation datasets
CMD_df = pd.merge(CMD_df, cmd_gene_Ann[['GeneID', 'Symbol']], on='GeneID', how='left')
CMD_df = CMD_df.set_index('Symbol')
CMD_df = CMD_df.drop('GeneID', axis=1)

In [ ]:
# Verify this was done correctly
CMD_df.head()

In [ ]:
# Remove version numbers from Geneid
PD_df["EnsemblGeneID"] = PD_df["Geneid"].str.split(".").str[0]  # Rename here

# Ensure GeneID is a string in both DataFrames
PD_df["EnsemblGeneID"] = PD_df["EnsemblGeneID"].astype(str)
cmd_gene_Ann["EnsemblGeneID"] = cmd_gene_Ann["EnsemblGeneID"].astype(str)

# Perform the merge
PD_df = pd.merge(PD_df, cmd_gene_Ann[['EnsemblGeneID', 'Symbol']], on='EnsemblGeneID', how='left')

# Set index to Symbol and drop GeneID
# df_pd = df_pd.rename(columns={'_id': 'Geneid'})
# PD_df = PD_df.merge(cmd_gene_Ann, on='Geneid', how='left')

In [ ]:
# Drop Geneid column from the AMP PD dataset and view
# PD_df = PD_df.set_index('Symbol', drop=True)
PD_df = PD_df.drop(columns=["Geneid"])
PD_df.head(500)

In [ ]:
# Check dimensions
print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)


In [ ]:
# Get intersection of common genes between the datasets
common_genes = df1.index.intersection(df2.index)

# Subset both datasets to common genes
df1 = df1.loc[common_genes]
df2 = df2.loc[common_genes]

In [ ]:
# Concatenate and get shape of the full dataset
merged_counts = pd.concat([df1, df2], axis=1)
print("Merged dataset shape:", merged_counts.shape)

In [ ]:
# Read AMP PD metadata
PD_metadata = pd.read_csv("sysbio/Transcriptomics/Blood/PD/releases_2023_v4release_1027_amp_pd_case_control.csv")

,participant_id,diagnosis_at_baseline,diagnosis_latest,case_control_other_at_baseline,case_control_other_latest
0,BF-1001,No PD Nor Other Neurological Disorder,No PD Nor Other Neurological Disorder,Control,Control
1,BF-1002,Idiopathic PD,Idiopathic PD,Case,Case
2,BF-1003,Idiopathic PD,Idiopathic PD,Case,Case
3,BF-1004,Idiopathic PD,Idiopathic PD,Case,Case
4,BF-1005,No PD Nor Other Neurological Disorder,No PD Nor Other Neurological Disorder,Control,Control


In [ ]:
# Subset AMP PD metadata to needed columns
PD_metadata = PD_metadata[["participant_id", "case_control_other_at_baseline"]]

pd_columns = list(PD_df.columns)

,participant_id,case_control_other_at_baseline
0,BF-1001,Control
1,BF-1002,Case
2,BF-1003,Case
3,BF-1004,Case
4,BF-1005,Control


In [ ]:
# Process strings to get needed samples
processed_strings = ["-".join(s.split("-")[:2]) for s in pd_columns]
print(processed_strings)

['HB-PD_INVXJ837TXN', 'HB-PD_INVPM710NJH', 'HB-PD_INVZC874MMT', 'HB-PD_INVWV624HAC', 'HB-PD_INVPK123UPD', 'HB-PD_INVHH968RZZ', 'HB-PD_INVWX362DL7', 'HB-PD_INVMW322DFP', 'HB-PD_INVCT548DEW', 'HB-PD_INVUL535YVX', 'HB-PD_INVAV201WYD', 'HB-PD_INVKM762GER', 'HB-PD_INVJF352UPG', 'HB-PD_INVCT548DEW', 'HB-PD_INVDG518AMF', 'HB-PD_INVAN423JWN', 'HB-PD_INVMG411AZK', 'HB-PD_INVRV224UAG', 'HB-PD_INVXD751CUN', 'HB-PD_INVCN268UPG', 'HB-PD_INVYZ403ABE', 'HB-PD_INVPE171ERM', 'HB-PD_INVZF591WW3', 'HB-PD_INVTE998TK0', 'HB-PD_INVTK579XW5', 'HB-PD_INVXC273EFP', 'HB-PD_INVFL000FBG', 'HB-PD_INVMY952TRF', 'HB-PD_INVDY491VZH', 'HB-PD_INVGV809WE0', 'HB-PD_INVLM077KED', 'HB-PD_INVVY168ZP9', 'HB-PD_INVWX925JX3', 'HB-PD_INVYK689VP7', 'HB-PD_INVBB462PV9', 'HB-PD_INVZA054JF4', 'HB-PD_INVRD245ZDR', 'HB-PD_INVXH634NAU', 'HB-PD_INVGW929KM2', 'HB-PD_INVUD383EXF', 'HB-PD_INVLL564YGH', 'HB-PD_INVTA736CUG', 'HB-PD_INVNU085CP9', 'HB-PD_INVYA693JP9', 'HB-PD_INVKN359MYE', 'HB-PD_INVRZ126CZ1', 'HB-PD_INVPJ782GYM', 'HB-PD_INVHX

In [ ]:
# Filter metadata_df to keep only rows matching AMP PD df columns
filtered_metadata_df = PD_metadata[PD_metadata['participant_id'].isin(processed_strings)]
filtered_metadata_df = filtered_metadata_df.rename(columns={"participant_id": "sample_id", "case_control_other_at_baseline": "group"})
filtered_metadata_df

In [ ]:
# Create new metadata df for AMP CMD
import pandas as pd

samples = list(CMD_df.columns)

cmd_meta = pd.DataFrame({
    "sample_id": samples,
    "group": [
        np.nan if s.startswith("G") else  
        "Control" if "U" in s else 
        "Treatment" if "T" in s else 
        np.nan
        for s in samples
    ]
})
cmd_meta

In [ ]:
# Combine AMP CMD metadata
metadata1 = cmd_meta
metadata2 = filtered_metadata_df
metadata = pd.concat([metadata1, metadata2])

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Read merged counts and transpose
merged_counts = pd.read_csv(merged_counts, index_col=0) 
data_transposed = merged_counts.T

# z-score normalize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_transposed)

# Fit PCA on the data and transform
pca = PCA(n_components=2)
pca_result = pca.fit_transform(data_scaled)

# Create PCA dataframe and merge with metadata
pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
pca_df["Sample"] = data_transposed.index  # Add sample names
pca_df = pca_df.merge(metadata, on="Sample")

# Plot PCs
fig_pca = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Disease",
    hover_data=["Sample"],
    title="PCA of Merged Transcriptomics Data",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)", 
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)"
    },
    template="plotly_white"
)
fig_pca.show()

# Isolate the 50 most variable genes
top_genes = merged_counts.var(axis=1).nlargest(50).index
top_gene_counts = merged_counts.loc[top_genes]

# Plot a heatmap of the 50 most variable genes
plt.figure(figsize=(12, 8))
sns.heatmap(
    top_gene_counts, cmap="viridis",
    yticklabels=True, xticklabels=False
)
plt.title("Heatmap of Top 50 Most Variable Genes")
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# Read in merged counts and metadata, set Sample column as the index
merged_counts = pd.read_csv("merged_counts.csv", index_col=0)  
metadata = pd.read_csv("metadata.csv")  
metadata = metadata.set_index("Sample")
merged_counts = merged_counts[metadata.index]  

# List unique diseases, throw an error if there are more than 2
unique_diseases = metadata["Disease"].unique()

if len(unique_diseases) != 2:
    raise ValueError("This script supports only two-group comparisons (e.g., Disease_A vs. Disease_B)")

# Label diseases and controls
case_label, control_label = unique_diseases
group_labels = (metadata["Disease"] == case_label).astype(int)  
log_counts = np.log2(merged_counts + 1)  

# Initialize lists to store values
p_values, logFC_values, beta_intercepts = [], [], []

X = sm.add_constant(group_labels.values)

# Loop through genes to get appropriate stats
for gene in log_counts.index:
    y = log_counts.loc[gene, :]
    model = sm.OLS(y, X).fit()
    
    beta_intercepts.append(model.params[0])  
    logFC_values.append(model.params[1]) 
    p_values.append(model.pvalues[1]) 

# Adjust p-values using multiple test correction
adj_p_values = multipletests(p_values, method="fdr_bh")[1]

# Build new dataframe from the lists
results_df = pd.DataFrame({
    "Gene": log_counts.index,
    "Beta_Intercept (β0)": beta_intercepts,
    "Beta_Case (β1)": logFC_values,  # logFC (effect size)
    "P_Value": p_values,
    "Adj_P_Value": adj_p_values
})

# Filter significant DEGs (FDR < 0.05)
significant_degs = results_df[results_df["Adj_P_Value"] < 0.05]

In [ ]:
import pandas as pd

# Sample metadata dataframe (Assuming it's loaded from a CSV or DataFrame)
sample_meta = pd.read_csv("sample_metadata.csv")  # Uncomment if reading from a CSV

# Filter Case samples based on study and visit
sample_case = sample_meta[
    (sample_meta["study"].isin(setStudy)) &
    (sample_meta["visit_month"].isin(setMonth)) &
    (sample_meta["case_control_other_at_baseline"] == "Case")
]
print(sample_case.shape)  # (1370, 63)


# Filter Control samples based on study and visit
sample_control = sample_meta[
    (sample_meta["study"].isin(setStudy)) &
    (sample_meta["visit_month"].isin(setMonth)) &
    (sample_meta["case_control_other_at_baseline"] == "Control")
]
print(sample_control.shape)  # (954, 63)


# Combine Case and Control tables
sample_meta = pd.concat([sample_case, sample_control], ignore_index=True)
print(sample_meta.shape)  # (2324, 63)

# Summary of the Cohort
print("How many unique sample IDs do we have?")
print(sample_meta["sample_id"].nunique())

print("Which studies are these sample IDs from?")
print(sample_meta["study"].value_counts())

print("How many case and control samples are there at baseline?")
print(sample_meta["case_control_other_at_baseline"].value_counts())

print("How many case and control samples are there at the latest visit? Is there a change between baseline and latest?")
print(sample_meta["case_control_other_latest"].value_counts())

print("How many samples have known LRRK2 mutations?")
print(sample_meta["has_known_LRRK2_mutation_in_WGS"].value_counts())

print("What are the dimensions of the final, merged sample table")
print(sample_meta.shape)

print("Show headers of the sample table")
print(sample_meta.head())

# Save the final merged table
sample_meta.to_csv("filtered_sample_meta.csv", index=False)
